# 🈶 ElearningJP: Huấn luyện Mô hình Nhận dạng 3.169 lớp Chữ viết tay (Kanji + Kana + Chữ cái / Số)

**Mục tiêu**: Huấn luyện mạng học sâu **Deep CNN với Residual Connections** có khả năng nhận diện xuất sắc:
1. **2.965 chữ Hán (Kanji)** thường dùng từ JLPT N5 đến N1.
2. **Hệ thống chữ Kana** (Hiragana & Katakana).
3. **52 chữ cái Latinh** (A-Z, a-z) và **10 chữ số** (0-9).

---

### ⚙️ Hướng dẫn cài đặt cấu hình trên Kaggle:
1. Ở bảng điều khiển bên phải (Notebook Options):
   * **Accelerator**: Chọn **GPU T4 x2** (hoặc GPU T4 x1).
   * **Internet**: Chọn **Internet On** (để tải thư viện và font chữ).
2. Thêm dữ liệu (Add Input):
   * Bấm **+ Add Input** ở góc trên bên phải.
   * Tìm kiếm dataset: `kanji-etl9` hoặc dataset ETL9G của bạn.
   * Đường dẫn mong đợi: `/kaggle/input/kanji-etl9/ETL9G_IMG` (hoặc `/kaggle/input/kanji-etl9`).


### 📌 Bước 1: Khởi tạo môi trường & Kiểm tra GPU


In [ ]:
import os
import gc
import json
import shutil
import random
import numpy as np
import tensorflow as tf
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ Đã kích hoạt {len(gpus)} GPU:")
    for gpu in gpus:
        print("  -", gpu.name)
else:
    print("⚠️ Cảnh báo: Chưa bật GPU! Hãy vào Notebook Options -> Accelerator -> chọn GPU T4.")


### 📌 Bước 2: Cấu hình thông số & Định nghĩa 133 lớp bổ sung


In [ ]:
# Cấu hình siêu tham số (Hyperparameters)
IMG_SIZE = (128, 128)
BATCH_SIZE = 64
SEED = 1203

# Tìm đường dẫn dataset ETL9G
POSSIBLE_PATHS = [
    '/kaggle/input/kanji-etl9/ETL9G_IMG',
    '/kaggle/input/kanji-etl9',
    '/kaggle/input/etl9g-images/ETL9G_IMG',
]

DATA_PATH = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if not DATA_PATH:
    raise FileNotFoundError("Không tìm thấy dataset ETL9G! Hãy kiểm tra lại thư mục input trong Kaggle.")

print(f"✅ Thư mục dữ liệu ETL9G: {DATA_PATH}")

# Danh sách 133 lớp bổ sung: Chữ cái in hoa, in thường, chữ số và Katakana biến âm
UPPER_LETTERS = [chr(c) for c in range(ord('A'), ord('Z') + 1)]
LOWER_LETTERS = [chr(c) for c in range(ord('a'), ord('z') + 1)]
DIGITS = [str(d) for d in range(10)]
KATAKANA_EXTENDED = [
    'ガ', 'ギ', 'グ', 'ゲ', 'ゴ',
    'ザ', 'ジ', 'ズ', 'ゼ', 'ゾ',
    'ダ', 'ヂ', 'ヅ', 'デ', 'ド',
    'バ', 'ビ', 'ブ', 'ベ', 'ボ',
    'パ', 'ピ', 'プ', 'ペ', 'ポ',
    'ヴ', 'ア', 'イ', 'ウ', 'エ', 'オ',
    'カ', 'キ', 'ク', 'ケ', 'コ',
    'サ', 'シ', 'ス', 'セ', 'ソ',
    'タ', 'チ', 'ツ', 'テ', 'ト',
    'ナ', 'ニ', 'ヌ', 'ネ', 'ノ',
    'ハ', 'ヒ', 'フ', 'ヘ', 'ホ',
    'マ', 'ミ', 'ム', 'メ', 'モ',
    'ヤ', 'ユ', 'ヨ',
    'ラ', 'リ', 'ル', 'レ', 'ロ',
    'ワ', 'ヲ', 'ン'
]

# Tổng hợp danh sách ký tự bổ sung (loại trừ trùng lặp)
ADDITIONAL_CHARS = list(dict.fromkeys(UPPER_LETTERS + LOWER_LETTERS + DIGITS + KATAKANA_EXTENDED))
print(f"Số lượng lớp bổ sung cần sinh mẫu: {len(ADDITIONAL_CHARS)}")


### 📌 Bước 3: Tự động sinh dữ liệu mẫu viết tay (Synthetic Balanced Dataset)
Để mô hình **không bị mất cân bằng (Class Imbalance)** và **không quên chữ Hán**, mỗi lớp bổ sung (chữ cái/số) sẽ được tự động sinh khoảng **150 ảnh mẫu** với các góc xoay, nét bút đậm/nhạt, dịch chuyển và nhiễu mô phỏng người viết tự nhiên.


In [ ]:
WORKING_DIR = Path('/kaggle/working/combined_dataset')
if WORKING_DIR.exists():
    shutil.rmtree(WORKING_DIR)
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# 1. Tạo symlink cho các thư mục chữ Kanji/Kana gốc từ ETL9G
print("Đang tạo liên kết dữ liệu ETL9G gốc...")
etl_classes = sorted([d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))])
for c in etl_classes:
    src_dir = os.path.join(DATA_PATH, c)
    dst_dir = WORKING_DIR / c
    try:
        os.symlink(src_dir, dst_dir)
    except:
        # Nếu không hỗ trợ symlink, copy thư mục
        shutil.copytree(src_dir, dst_dir)

print(f"✅ Đã nạp {len(etl_classes)} lớp từ ETL9G.")

# 2. Sinh dữ liệu viết tay mô phỏng cho các ký tự bổ sung
print("Đang sinh dữ liệu viết tay cho các ký tự chữ cái, số, Kana...")
SAMPLES_PER_CHAR = 160

# Tải font chữ chuẩn tiếng Nhật / tiếng Anh
os.system("apt-get update -qq && apt-get install -y -qq fonts-noto-cjk fonts-dejavu-core")

FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
]
available_fonts = [f for f in FONT_PATHS if os.path.exists(f)]

def generate_handwritten_sample(char, font_path, size=128):
    """Tạo 1 ảnh nền đen nét trắng mô phỏng chữ viết tay."""
    img = Image.new('L', (size, size), color=0)
    draw = ImageDraw.Draw(img)
    
    font_size = random.randint(70, 95)
    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        font = ImageFont.load_default()
        
    # Tính tọa độ căn giữa
    bbox = draw.textbbox((0, 0), char, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    
    offset_x = (size - w) // 2 + random.randint(-8, 8)
    offset_y = (size - h) // 2 + random.randint(-8, 8)
    
    draw.text((offset_x, offset_y), char, fill=255, font=font)
    
    # Biến dạng xoay nhẹ & co giãn
    angle = random.uniform(-12, 12)
    img = img.rotate(angle, resample=Image.BILINEAR, expand=False)
    
    # Làm mờ nhẹ nét bút và tăng độ dày
    if random.random() > 0.4:
        img = img.filter(ImageFilter.MaxFilter(random.choice([3, 5])))
    if random.random() > 0.5:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 0.8)))
        
    return img

generated_count = 0
for char in ADDITIONAL_CHARS:
    # Đặt tên thư mục theo mã Hex hoặc ký tự an toàn
    folder_name = f"char_{ord(char):04x}"
    char_dir = WORKING_DIR / folder_name
    if char_dir.exists():
        continue
    char_dir.mkdir(parents=True, exist_ok=True)
    
    font_choice = available_fonts[0] if available_fonts else None
    for idx in range(SAMPLES_PER_CHAR):
        f = random.choice(available_fonts) if available_fonts else None
        sample_img = generate_handwritten_sample(char, f, size=128)
        sample_img.save(char_dir / f"sample_{idx:04d}.png")
        
    generated_count += 1

all_classes = sorted([d for d in os.listdir(WORKING_DIR) if os.path.isdir(WORKING_DIR / d)])
print(f"✅ Hoàn tất! Tổng số lớp phân loại trong tập dữ liệu kết hợp: {len(all_classes)} lớp.")


### 📌 Bước 4: Tạo Pipeline dữ liệu huấn luyện & kiểm định (Train / Val Split 80:20)


In [ ]:
# 1. Tải tập Train và Validation
train_ds = tf.keras.utils.image_dataset_from_directory(
    WORKING_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    WORKING_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale'
)

# 2. Xuất ngay file từ điển nhãn (Labels JSON) đúng theo thứ tự phân loại của Dataset
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"Tổng số lớp phân loại: {NUM_CLASSES}")

# Giải mã tên thư mục thành ký tự thật
final_labels = []
for c in class_names:
    if c.startswith("char_"):
        try:
            final_labels.append(chr(int(c.replace("char_", ""), 16)))
        except:
            final_labels.append(c)
    elif c.startswith("0x") or c.startswith("0X"):
        try:
            final_labels.append(chr(int(c, 16)))
        except:
            final_labels.append(c)
    else:
        final_labels.append(c)

LABEL_OUTPUT_PATH = '/kaggle/working/label_3169.json'
with open(LABEL_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(final_labels, f, ensure_ascii=False)
print(f"✅ Đã lưu danh sách nhãn chuẩn vào: {LABEL_OUTPUT_PATH}")

# 3. Pipeline Data Augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.06),
    tf.keras.layers.RandomZoom(0.06),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.GaussianNoise(0.003)
])

normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = (
    train_ds
    .shuffle(1024)
    .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    .map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)


### 📌 Bước 5: Xây dựng Kiến trúc Mạng Deep CNN Residual


In [ ]:
from tensorflow.keras import layers, models

def build_kanji_residual_cnn(num_classes, input_shape=(128, 128, 1)):
    inputs = layers.Input(shape=input_shape)

    # --- Block 1: Trích xuất góc cạnh nét ---
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_initializer='he_normal')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Conv2D(64, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # --- Block 2: Trích xuất bộ thủ (Radicals) ---
    x = layers.Conv2D(128, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Conv2D(128, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # --- Block 3: Khối Residual 256 filters ---
    x_shortcut = x
    x = layers.Conv2D(256, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Conv2D(256, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)

    x_shortcut = layers.Conv2D(256, (1, 1), padding='same', kernel_initializer='he_normal')(x_shortcut)
    x = layers.Add()([x, x_shortcut])
    x = layers.LeakyReLU(0.1)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # --- Block 4: Khối Residual 384 filters ---
    x_shortcut = x
    x = layers.Conv2D(384, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Conv2D(384, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)

    x_shortcut = layers.Conv2D(384, (1, 1), padding='same', kernel_initializer='he_normal')(x_shortcut)
    x = layers.Add()([x, x_shortcut])
    x = layers.LeakyReLU(0.1)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # --- Tầng phân loại (Classification Head) ---
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(256, kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="Kanji_Residual_CNN_3169")
    return model

model = build_kanji_residual_cnn(num_classes=NUM_CLASSES)
print(f"✅ Tạo mô hình thành công. Tổng số tham số: {model.count_params():,}")


### 📌 Bước 6: Huấn luyện mô hình với Callbacks & Learning Rate Schedule


In [ ]:
MODEL_SAVE_PATH = '/kaggle/working/kanji_model_extended_v2.h5'

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name='top1_acc'),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5_acc')
    ]
)

cb_list = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor='val_top1_acc',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_top1_acc',
        patience=5,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1,
        min_lr=1e-6
    )
]

print("Bắt đầu huấn luyện mô hình...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=16,
    callbacks=cb_list,
    verbose=1
)

print(f"\n🎉 Huấn luyện hoàn tất! Mô hình tốt nhất đã lưu tại: {MODEL_SAVE_PATH}")


### 📌 Bước 7: Đánh giá đồ thị độ chính xác & Nén file tải về máy


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Đồ thị Accuracy
ax1.plot(history.history['top1_acc'], label='Train Top-1 Acc')
ax1.plot(history.history['val_top1_acc'], label='Val Top-1 Acc')
ax1.plot(history.history['val_top5_acc'], label='Val Top-5 Acc', linestyle='--')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# Đồ thị Loss
ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_results.png', dpi=200)
plt.show()

# Đóng gói file model và file nhãn thành file ZIP để tải về máy tiện lợi
os.system("zip -j /kaggle/working/kanji_model_release.zip /kaggle/working/kanji_model_extended_v2.h5 /kaggle/working/label_3169.json /kaggle/working/training_results.png")
print("✅ Đã tạo file: /kaggle/working/kanji_model_release.zip")
print("👉 Bạn chỉ cần bấm vào file kanji_model_release.zip ở mục Output bên phải màn hình Kaggle và chọn Download!")
